In [13]:
import numpy as np
import inspect
import importlib
import Utils
importlib.reload(Utils)
from rasterstats import zonal_stats
import Constants
importlib.reload(Constants)
import ConstantObjects
importlib.reload(ConstantObjects)
import matplotlib.pyplot as plt
import os
import rasterio
from rasterio.warp import reproject, Resampling

print("Done with cell!")



[Line 13] n_cols in ConstantObjects: 21
[Line 15] n_cols: 21, n_rows: 18
[Line 26] n_cols in ConstantObjects: 21
[Line 28] gdf_tree_circles.shape: (378, 1)
x0 =  -10783454.50783975
y0 =  2246774.099946275
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (19.777898, -96.869939)
x0 =  -10783454.50783975
y0 =  2246774.099946275
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (19.777882, -96.869634)
x0 =  -10783576.959279623
y0 =  2246703.1213188255
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (19.777201, -96.871090)
Constants.n_cols =  21
Constants.n_rows =  18
x0 =  -10783454.50783975
y0 =  2246395.5503548854
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (19.778652, -96.863761)
x0 =  -10783454.50783975
y0 =  2246774.099946275
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (19.777761, -96.869698)
x0 =  -10783398.848094355
y0 =  2246927.887889348
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Cente

In [14]:
#dates = Utils.generate_date_range("2025-01-25", "2025-01-26",  "%Y-%m-%d", 5)

#NDMI = Normalized Difference Moisture Index
# Output lists
ndmi_values = []
valid_dates = []

output_tif_cumulative = "ndmi_cumulative.tif"
output_tif = ""
gndvi_tif = ""
rgb_tif = ""

mgrs_tile = Utils.latlon_to_s2_tile(
    Constants.lat0,
    Constants.lon0
)

print(
    "RUN LOCATION:",
    Constants.lat0,
    Constants.lon0,
    "tile =", mgrs_tile
)

assert mgrs_tile == "14QQG", (
    f"Unexpected tile {mgrs_tile} "
    f"for lat={Constants.lat0}, lon={Constants.lon0}"
)
print("Done with cell!")

[Line 203] ... Mapped latitude  19.7782 , longitude  -96.86942  to mgrs_tile =  14QQG
RUN LOCATION: 19.7782 -96.86942 tile = 14QQG
Done with cell!


In [ ]:
# Re-run the previous code after reset
#veracruz good dates
#2024-01-13
#2025-01-12


dates = Utils.generate_date_range("2022-01-01", "2026-08-18", Constants.lat0, Constants.lon0, "%Y-%m-%d")

print("dates = ",dates)


# Sample NDMI at the given point
for date in dates:
    print("processing date ",date)
    # Download images corresponding to bands 2,3,4,8,11.
    b02_path = Utils.download_band_dynamic(date, "B02",Constants.lat0,Constants.lon0)  # Blue
    b03_path = Utils.download_band_dynamic(date, "B03",Constants.lat0,Constants.lon0)  # Green
    b04_path = Utils.download_band_dynamic(date, "B04",Constants.lat0,Constants.lon0)  # Red
    b08_path = Utils.download_band_dynamic(date, "B08",Constants.lat0,Constants.lon0)
    print(f"[Cell Line {inspect.currentframe().f_lineno}] ...  ")
    b11_path = Utils.download_band_dynamic(date, "B11",Constants.lat0,Constants.lon0)
    print(f"Cell [Line {inspect.currentframe().f_lineno}] ...  ")
    #b04_path = Utils.download_band_dynamic(date, "B04",Constants.lat0,Constants.lon0)
    
    print("set b02_path ",  b02_path)
    print("set b03_path ",  b03_path)
    print("set b04_path ",  b04_path)
    print("set b08_path ",  b08_path)
    print(f"Cell [Line {inspect.currentframe().f_lineno}] ...  ")
    print("set b11_path ",  b11_path)

    
    print(f"Cell [Line {inspect.currentframe().f_lineno}] ... About to try ")
    try:
        with rasterio.open(b08_path) as src_b08, rasterio.open(b11_path) as src_b11, rasterio.open(b03_path) as src_b03 , rasterio.open(b04_path) as src_b04 : # , rasterio.open(b02_path) as src_b02, rasterio.open(b03_path) as src_b03, rasterio.open(b04_path) as src_b04:
            print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")

            b11_data = src_b11.read(1)
            print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")

            b03_data = src_b03.read(1)
            b04_data = src_b04.read(1)
            print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")

            resampled_b11 = np.empty((10980, 10980), dtype="float32")
            # Create an ndmi tif for this date
            output_tif = Utils.make_path_name("s2_derived_tifs", mgrs_tile,date, "ndmi","tif")
            os.path.exists(output_tif) or (Utils.write_ndmi_geotiff(  b08_path, b11_path, output_tif), print("✅ NDMI GeoTIFF in UTM saved to:", output_tif))
            out_dtype = "uint8"
            print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")

            rgb_tif    = Utils.make_path_name("s2_derived_tifs", mgrs_tile,date, "rgb."+out_dtype,"tif")
            os.path.exists(rgb_tif) or (Utils.write_rgb_geotiff_3857(b02_path, b03_path, b04_path, rgb_tif, out_dtype), print("✅ RGB GeoTIFF in UTM saved to:", rgb_tif))
            #out_dtype = "float32"
            #rgb_tif    = Utils.make_path_name("s2_derived_tifs", mgrs_tile,date, "rgb."+out_dtype,"tif")
            #os.path.exists(rgb_tif) or (Utils.write_rgb_geotiff_3857(b02_path, b03_path, b04_path, rgb_tif, out_dtype), print("✅ RGB GeoTIFF in UTM saved to:", rgb_tif) )    
            #out_dtype = "float32"
            print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")

            gndvi_tif = Utils.make_path_name("s2_derived_tifs", mgrs_tile,date, "gndvi","tif")
            print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")
            os.path.exists(gndvi_tif) or (Utils.write_gndvi_geotiff(  b08_path, b03_path, gndvi_tif), print("✅ GNDVI GeoTIFF in UTM saved to:", gndvi_tif))
            print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")
                
    except Exception as e:
        print(f"Skipping {date} due to error: {e}")

print("output_tif =", repr(output_tif))
print("exists =", os.path.exists(output_tif) if output_tif else False)

print(f"Cell [Line {inspect.currentframe().f_lineno}] ... Done with cell! ")


[Line 203] ... Mapped latitude  19.7782 , longitude  -96.86942  to mgrs_tile =  14QQG
First acquisition: 2022-01-03
dates =  ['2022-01-03', '2022-01-08', '2022-01-13', '2022-01-18', '2022-01-23', '2022-01-28', '2022-02-02', '2022-02-07', '2022-02-12', '2022-02-17', '2022-02-22', '2022-02-27', '2022-03-04', '2022-03-09', '2022-03-14', '2022-03-19', '2022-03-24', '2022-03-29', '2022-04-03', '2022-04-08', '2022-04-13', '2022-04-18', '2022-04-23', '2022-04-28', '2022-05-03', '2022-05-08', '2022-05-13', '2022-05-18', '2022-05-23', '2022-05-28', '2022-06-07', '2022-06-12', '2022-06-17', '2022-06-22', '2022-06-27', '2022-07-02', '2022-07-07', '2022-07-12', '2022-07-17', '2022-07-22', '2022-07-27', '2022-08-01', '2022-08-06', '2022-08-11', '2022-08-16', '2022-08-21', '2022-08-26', '2022-08-31', '2022-09-05', '2022-09-10', '2022-09-15', '2022-09-20', '2022-09-25', '2022-09-30', '2022-10-05', '2022-10-10', '2022-10-15', '2022-10-20', '2022-10-25', '2022-10-30', '2022-11-04', '2022-11-09', '2022-

/var/folders/dm/6jp_xjys25gdd27bjkbvnhk80000gn/T/ipykernel_41331/2670316693.py:38: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  b11_data = src_b11.read(1)


Cell [Line 39] ... 


/var/folders/dm/6jp_xjys25gdd27bjkbvnhk80000gn/T/ipykernel_41331/2670316693.py:41: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  b03_data = src_b03.read(1)
/var/folders/dm/6jp_xjys25gdd27bjkbvnhk80000gn/T/ipykernel_41331/2670316693.py:42: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  b04_data = src_b04.read(1)


Cell [Line 43] ... 
band = > ndmi <
extension = > tif <
Cell [Line 50] ... 
band = > rgb.uint8 <
extension = > tif <
Cell [Line 58] ... 
band = > gndvi <
extension = > tif <
Cell [Line 61] ... 
Cell [Line 63] ... 
processing date  2022-01-08
[Line 203] ... Mapped latitude  19.7782 , longitude  -96.86942  to mgrs_tile =  14QQG
[Line 203] ... Mapped latitude  19.7782 , longitude  -96.86942  to mgrs_tile =  14QQG
[Line 203] ... Mapped latitude  19.7782 , longitude  -96.86942  to mgrs_tile =  14QQG
[Line 203] ... Mapped latitude  19.7782 , longitude  -96.86942  to mgrs_tile =  14QQG
[Cell Line 20] ...  
[Line 203] ... Mapped latitude  19.7782 , longitude  -96.86942  to mgrs_tile =  14QQG
Cell [Line 22] ...  
set b02_path  s2_point_series/14QQG_2022-01-08_B02.jp2
set b03_path  s2_point_series/14QQG_2022-01-08_B03.jp2
set b04_path  s2_point_series/14QQG_2022-01-08_B04.jp2
set b08_path  s2_point_series/14QQG_2022-01-08_B08.jp2
Cell [Line 29] ...  
set b11_path  s2_point_series/14QQG_2022-01-0

...

Cell [Line 39] ... 


/var/folders/dm/6jp_xjys25gdd27bjkbvnhk80000gn/T/ipykernel_41331/2670316693.py:41: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  b03_data = src_b03.read(1)
/var/folders/dm/6jp_xjys25gdd27bjkbvnhk80000gn/T/ipykernel_41331/2670316693.py:42: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  b04_data = src_b04.read(1)


Cell [Line 43] ... 
band = > ndmi <
extension = > tif <
Cell [Line 50] ... 
band = > rgb.uint8 <
extension = > tif <
Cell [Line 58] ... 
band = > gndvi <
extension = > tif <
Cell [Line 61] ... 
Cell [Line 63] ... 
processing date  2022-04-28
[Line 203] ... Mapped latitude  19.7782 , longitude  -96.86942  to mgrs_tile =  14QQG
[Line 203] ... Mapped latitude  19.7782 , longitude  -96.86942  to mgrs_tile =  14QQG
[Line 203] ... Mapped latitude  19.7782 , longitude  -96.86942  to mgrs_tile =  14QQG
[Line 203] ... Mapped latitude  19.7782 , longitude  -96.86942  to mgrs_tile =  14QQG
[Cell Line 20] ...  
[Line 203] ... Mapped latitude  19.7782 , longitude  -96.86942  to mgrs_tile =  14QQG
Cell [Line 22] ...  
set b02_path  s2_point_series/14QQG_2022-04-28_B02.jp2
set b03_path  s2_point_series/14QQG_2022-04-28_B03.jp2
set b04_path  s2_point_series/14QQG_2022-04-28_B04.jp2
set b08_path  s2_point_series/14QQG_2022-04-28_B08.jp2
Cell [Line 29] ...  
set b11_path  s2_point_series/14QQG_2022-04-2

/var/folders/dm/6jp_xjys25gdd27bjkbvnhk80000gn/T/ipykernel_41331/2670316693.py:38: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  b11_data = src_b11.read(1)


Cell [Line 39] ... 


In [ ]:
import leafmap
#importlib.reload(ConstantObjects)
print(f"[Line {inspect.currentframe().f_lineno}] ... lat0 = ",Constants.lat0)
print(f"[Line {inspect.currentframe().f_lineno}] ... lon0 = ",Constants.lon0)

m2 = leafmap.Map(basemap="Esri.WorldImagery")
#m2 = leafmap.Map(center=(Constants.lat0,Constants.lon0), zoom=15)
m2.add_raster(output_tif, layer_name="NDMI", opacity=0.9, nodata=float("nan"))

m2.set_center(Constants.lon0, Constants.lat0,  zoom=15)

            
m2.add_gdf(ConstantObjects.gdf_box_terreno_casa, layer_name="Plot Boundary")
m2.add_gdf(ConstantObjects.gdf_box_plots_4_5, layer_name="Plots 4,5")
m2.add_gdf(ConstantObjects.gdf_box_pueblo, layer_name="pueblo")
m2.add_gdf(ConstantObjects.gdf_box_river, layer_name="river")

#show the entire ndmi tile
#m2.add_raster(output_tif_cumulative, layer_name="NDMI", colormap="BrBG", nodata=np.nan, opacity = .55)
#print(f"[Line {inspect.currentframe().f_lineno}] ... b04_path = ",b04_path)
#m2.add_raster(b04_path, layer_name="Red", colormap="BrBG", nodata=np.nan, opacity = .9)



#ndmi_3857 = reproject_to_3857("ndmi_14QQG.tif", "ndmi_14QQG_3857.tif")
#m2.add_raster(output_tif, layer_name="NDMI", opacity=0.9)

#m2.add_raster(ndmi_3857, layer_name="NDMI", opacity=0.9, nodata=float("nan"))


m2